In [93]:
import json

from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index
from anthropic import Anthropic
from pydantic import BaseModel

In [94]:
client = Anthropic()

In [95]:
reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

In [96]:
parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

In [97]:
index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

print(f"Indexed {len(chunked_docs)} chunks from {len(files)} documents")

Indexed 382 chunks from 95 documents


In [98]:
def search(query: str):
    results = index.search(
        query=query,
        num_results=5
    )
    return results

In [99]:
instructions = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question: str, search_results: list[dict]) -> str:
    context = json.dumps(search_results, indent=2)
    return prompt_template.format(
        question=question,
        context=context
    )

In [100]:
def llm(
    prompt: str,
    instructions: str | None = None,
    model: str = "claude-opus-5",
    max_tokens: int = 2056,
) -> str:
    response = client.messages.create(
        model=model,
        max_tokens=max_tokens,
        messages=[
            {"role": "assistant", "content": instructions or "You are a helpful course assistant."},
            {"role": "user", "content": prompt}
        ],
        thinking={"type": "adaptive", "display": "summarized"},
    )
    for block in response.content:
        if block.type == "text":
            return block.text

In [101]:
def rag(query: str):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm(prompt, instructions)

In [102]:
answer = rag('how do I implement LLM as a judge?')
print("Answer:", answer[:100])

Answer: ## Implementing an LLM as a Judge (Evidently)

Based on the documentation tutorial, here's the workf


In [103]:
def llm_structured(
    user_prompt: str,
    output_type: type[BaseModel],
    instructions: str | None = None,
    model: str = "claude-opus-5",
    max_tokens: int = 2056,
) -> dict:
    response = client.messages.parse(
        model="claude-opus-5-5",
        max_tokens=max_tokens,
        messages=[
            {"role": "assistant", "content": instructions or "You are a helpful course assistant."},
            {"role": "user", "content": user_prompt},
        ],
        output_format=output_type,
    )
    return response.parsed_output

In [104]:
class CalendarEvent(BaseModel):
    name: str
    date: str
    participants: list[str]

In [105]:
response = llm_structured(
    instructions="Extract the event information.",
    user_prompt="Alice and Bob are going to a science fair on Friday.",
    output_type=CalendarEvent,
)

In [106]:
response

CalendarEvent(name='Science Fair', date='Friday', participants=['Alice', 'Bob'])

In [107]:
class RAGResponse(BaseModel):
    answer: str
    found_answer: bool

In [108]:
RAGResponse.model_json_schema()

{'properties': {'answer': {'title': 'Answer', 'type': 'string'},
  'found_answer': {'title': 'Found Answer', 'type': 'boolean'}},
 'required': ['answer', 'found_answer'],
 'title': 'RAGResponse',
 'type': 'object'}

In [109]:
def rag_structured(query: str, output_type: type[BaseModel]):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm_structured(
        instructions=instructions,
        user_prompt=prompt,
        output_type=output_type,
    )

In [110]:
answer = rag_structured("how do I do llm evals?", output_type=RAGResponse)

print(answer.answer[:100])
print(answer.found_answer)

Based on the documentation, LLM evals in Evidently follow this general workflow:

**1. Install and i
True


In [111]:
answer = rag_structured("how do I install kafka on windows?", output_type=RAGResponse)

print(answer.answer[:100])
print(answer.found_answer)

Our documentation doesn't cover installing Kafka on Windows. It covers Evidently and its related pac
False


In [112]:
from typing import Optional


class RAGResponse(BaseModel):
    answer: Optional[str] = None
    found_answer: bool

In [113]:
RAGResponse.model_json_schema()

{'properties': {'answer': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
   'default': None,
   'title': 'Answer'},
  'found_answer': {'title': 'Found Answer', 'type': 'boolean'}},
 'required': ['found_answer'],
 'title': 'RAGResponse',
 'type': 'object'}

In [114]:
answer = rag_structured('how do I install kafka on windows?', RAGResponse)

print(answer.answer)
print(answer.found_answer)

The provided documentation doesn't cover installing Kafka on Windows. It only describes how to install Evidently: `pip install evidently` or `conda install -c conda-forge evidently`, `pip install evidently[llm]` for LLM-specific evaluations, and `pip install tracely` for OpenTelemetry-based tracing. For Kafka, please check the official Apache Kafka documentation.
False


In [115]:
instructions = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.

If you don't find the answer, set `answer` to None
"""

In [116]:
answer = rag_structured('how do I install kafka on windows?', RAGResponse)

print(answer.answer)
print(answer.found_answer)

None
False


In [117]:
class RAGResponse(BaseModel):
    """
    The response from the documentation RAG system

    If the answer to the question wasn't found in the database, `answer` is None
    """
    answer: Optional[str] = None
    found_answer: bool

In [118]:
from pydantic import Field


class RAGResponse(BaseModel):
    """
    The response from the documentation RAG system
    """
    answer: Optional[str] = Field(None, description="Answer to the question or None if it's not found")
    found_answer: bool = Field(description="True if the answer is found, False otherwise")

In [119]:
RAGResponse.model_json_schema()

{'description': 'The response from the documentation RAG system',
 'properties': {'answer': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
   'default': None,
   'description': "Answer to the question or None if it's not found",
   'title': 'Answer'},
  'found_answer': {'description': 'True if the answer is found, False otherwise',
   'title': 'Found Answer',
   'type': 'boolean'}},
 'required': ['found_answer'],
 'title': 'RAGResponse',
 'type': 'object'}

In [120]:
answer = rag_structured('how do I install kafka on windows?', RAGResponse)

print(answer.answer)
print(answer.found_answer)

None
False


In [121]:
from typing import Literal


class RAGResponse(BaseModel):
    """
    This model provides a structured answer with metadata about the response,
    including confidence, categorization, and follow-up suggestions.
    """

    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0 indicating how certain the answer is")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions the user might want to ask")

In [122]:
RAGResponse.model_json_schema()

{'description': 'This model provides a structured answer with metadata about the response,\nincluding confidence, categorization, and follow-up suggestions.',
 'properties': {'answer': {'description': "The main answer to the user's question in markdown",
   'title': 'Answer',
   'type': 'string'},
  'found_answer': {'description': 'True if relevant information was found in the documentation',
   'title': 'Found Answer',
   'type': 'boolean'},
  'confidence': {'description': 'Confidence score from 0.0 to 1.0 indicating how certain the answer is',
   'title': 'Confidence',
   'type': 'number'},
  'confidence_explanation': {'description': 'Explanation about the confidence level',
   'title': 'Confidence Explanation',
   'type': 'string'},
  'answer_type': {'description': 'The category of the answer',
   'enum': ['how-to',
    'explanation',
    'troubleshooting',
    'comparison',
    'reference'],
   'title': 'Answer Type',
   'type': 'string'},
  'followup_questions': {'description': 'S

In [124]:
answer = rag_structured('how do I evaluate llms', RAGResponse)

In [126]:
answer.model_dump()

{'answer': 'The documentation provided covers one specific way to evaluate LLM outputs with Evidently: **LLM-as-a-jury**. In this approach, several LLMs judge the same output. You can then combine their verdicts into one result (for example, an output passes only if all judges, or most of them, approve). You can also use it to surface cases where the judges disagree.\n\n### 1. Install and import\n```python\npip install evidently litellm\n```\n(Or install `evidently[llm]`.)\n\nThen import the components you need, such as `Dataset`, `DataDefinition`, `Report`, `TextEvals`, `LLMEval`, `TestSummary`, `BinaryClassificationPromptTemplate` and `CustomColumnDescriptor`.\n\n### 2. Set up the evaluator LLMs\nSet the API key for each judge:\n```python\nimport os\nos.environ["OPENAI_API_KEY"] = "YOUR KEY"\nos.environ["GEMINI_API_KEY"] = "YOUR KEY"\nos.environ["ANTHROPIC_API_KEY"] = "YOUR KEY"\n```\nYou can use other LLMs too, including self-hosted ones.\n\n**Optional:** create an Evidently Cloud w

In [129]:
answer = rag_structured('how do I install kafka on windows?', RAGResponse)

In [130]:
answer.model_dump()

{'answer': "The documentation provided doesn't contain any information about installing Kafka on Windows. The available docs cover installing **Evidently** (`pip install evidently` or `conda install -c conda-forge evidently`), the LLM extras (`pip install evidently[llm]`), and the **tracely** tracing package (`pip install tracely`). Kafka isn't mentioned anywhere.\n\nFor Kafka installation steps, please refer to the official Apache Kafka documentation.",
 'found_answer': False,
 'confidence': 0.95,
 'confidence_explanation': "The context only covers Evidently installation and platform evaluation features. Kafka isn't mentioned at all, so it is clear the answer is not in the documentation.",
 'answer_type': 'how-to',
 'followup_questions': ['How do I install Evidently using pip or conda?',
  'How do I install Evidently with LLM-specific dependencies?',
  'How do I install tracely for OpenTelemetry-based tracing?']}

In [131]:
from pydantic import model_validator


class AnswerNotFound(BaseModel):
    explanation: str


class AnswerResponse(BaseModel):
    """
    If answer is found, 'answer' is populated.
    If no answer is found, 'answer_not_found' is populated.
    Only one of the two fields can be set at a time. Never both or neither.
    """

    found_answer: bool
    answer: Optional[RAGResponse] = None
    answer_not_found: Optional[AnswerNotFound] = None

    @model_validator(mode="after")
    def check_consistency(self):
        if self.answer is not None and self.answer_not_found is not None:
            raise ValueError("Provide either 'answer' or 'answer_not_found', not both.")

        if self.answer is None and self.answer_not_found is None:
            raise ValueError("Provide either 'answer' or 'answer_not_found'.")

        return self

In [132]:
AnswerResponse.model_json_schema()

{'$defs': {'AnswerNotFound': {'properties': {'explanation': {'title': 'Explanation',
     'type': 'string'}},
   'required': ['explanation'],
   'title': 'AnswerNotFound',
   'type': 'object'},
  'RAGResponse': {'description': 'This model provides a structured answer with metadata about the response,\nincluding confidence, categorization, and follow-up suggestions.',
   'properties': {'answer': {'description': "The main answer to the user's question in markdown",
     'title': 'Answer',
     'type': 'string'},
    'found_answer': {'description': 'True if relevant information was found in the documentation',
     'title': 'Found Answer',
     'type': 'boolean'},
    'confidence': {'description': 'Confidence score from 0.0 to 1.0 indicating how certain the answer is',
     'title': 'Confidence',
     'type': 'number'},
    'confidence_explanation': {'description': 'Explanation about the confidence level',
     'title': 'Confidence Explanation',
     'type': 'string'},
    'answer_type': 

In [134]:
answer = rag_structured('how do I install kafka on windows?', AnswerResponse)
answer.model_dump()

{'found_answer': False,
 'answer': None,
 'answer_not_found': {'explanation': "The documentation provided doesn't cover installing Kafka on Windows or anywhere else. It covers Evidently (an AI/ML evaluation library) and related topics: installing Evidently, Evidently LLM and Tracely with pip or conda; running evaluations; synthetic data; and the Explore view. For Kafka installation, see the official Apache Kafka documentation."}}

In [135]:
answer = rag_structured('how do I run llm evals?', AnswerResponse)
answer.model_dump()

{'found_answer': True,
 'answer': {'answer': 'You run LLM evals with the Evidently Python library. You compute **descriptors** (output-level text evaluations) on your dataset, then summarize them in a Report using the `TextEvals` Preset. You can view the results locally or upload them to Evidently Cloud.\n\n### 1. Install and import\n```python\npip install evidently[llm]\n```\n```python\nimport pandas as pd\nfrom evidently.future.datasets import Dataset, DataDefinition\nfrom evidently.future.descriptors import *\nfrom evidently.future.report import Report\nfrom evidently.future.presets import TextEvals\n```\nIf you use LLM-as-a-judge evaluators, also set your OpenAI key:\n```python\nimport os\nos.environ["OPENAI_API_KEY"] = "YOUR KEY"\n```\n\n### 2. Create a Dataset with descriptors\n```python\neval_dataset = Dataset.from_pandas(\n    pd.DataFrame(df),\n    data_definition=DataDefinition(),\n    descriptors=[\n        TextLength("target_response", alias="Length"),\n        SentenceCoun

In [136]:
from pydantic import ValidationError


try:
    AnswerResponse()
except ValidationError as e:
    print("Validation error:")
    print(e)

Validation error:
1 validation error for AnswerResponse
found_answer
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
